# Name / SMILES → XYZ

Turn a molecule name or SMILES into 3D coordinates (xyz) to copy into an ORCA input.

**How to use:** run cell 1 once, then enter your molecule in cell 2 and run it with the ▶
button on its left. The coordinates and a 3D view appear below the cell. For the next
molecule, change it and run cell 2 again.

Part of the [TUS lecture](https://svatunek-lab.github.io/TUS_compchem_lecture/).

In [ ]:
#@title 1 · Install RDKit (run once, takes ~30 s)
!pip install -q rdkit py3Dmol
print("Done.")

In [ ]:
#@title 2 · Enter a molecule, then run this cell (▶) { display-mode: "form" }
#@markdown A **name** (e.g. `aspirin`, `2-butanone`) or a **SMILES** (e.g. `CC(=O)Oc1ccccc1C(=O)O`).
molecule = "aspirin"  #@param {type:"string"}
#@markdown `auto` tries SMILES first, then looks the name up in PubChem.
input_type = "auto"  #@param ["auto", "SMILES", "name"]

import urllib.parse
import urllib.request

import py3Dmol
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")


def name_to_smiles(name):
    """Look up a compound name in PubChem and return its SMILES."""
    quoted = urllib.parse.quote(name.strip())
    for prop in ("SMILES", "IsomericSMILES"):
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quoted}/property/{prop}/TXT"
        try:
            with urllib.request.urlopen(url, timeout=20) as r:
                text = r.read().decode().strip()
            if text:
                return text.splitlines()[0]
        except Exception:
            pass
    raise ValueError(f"'{name}' was not found in PubChem. Check the spelling or use a SMILES.")


def to_mol(text, kind):
    if kind in ("auto", "SMILES"):
        mol = Chem.MolFromSmiles(text.strip())
        if mol is not None:
            return mol, text.strip()
        if kind == "SMILES":
            raise ValueError(f"'{text}' is not a valid SMILES.")
    smiles = name_to_smiles(text)
    print(f"PubChem: {text} -> {smiles}")
    return Chem.MolFromSmiles(smiles), smiles


mol, smiles = to_mol(molecule, input_type)
mol = Chem.AddHs(mol)

# 3D structure: ETKDG embedding + force-field optimisation (MMFF, UFF as fallback)
params = AllChem.ETKDGv3()
params.randomSeed = 42
if AllChem.EmbedMolecule(mol, params) != 0:
    params.useRandomCoords = True
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError("Could not generate 3D coordinates for this molecule.")
if AllChem.MMFFHasAllMoleculeParams(mol):
    AllChem.MMFFOptimizeMolecule(mol, maxIters=2000)
else:
    AllChem.UFFOptimizeMolecule(mol, maxIters=2000)

charge = Chem.GetFormalCharge(mol)
multiplicity = sum(a.GetNumRadicalElectrons() for a in mol.GetAtoms()) + 1

conf = mol.GetConformer()
coords = "\n".join(
    f"{a.GetSymbol():<2} {conf.GetAtomPosition(a.GetIdx()).x:12.6f}"
    f" {conf.GetAtomPosition(a.GetIdx()).y:12.6f} {conf.GetAtomPosition(a.GetIdx()).z:12.6f}"
    for a in mol.GetAtoms()
)

print(f"{mol.GetNumAtoms()} atoms, charge {charge}, multiplicity {multiplicity}")
print("Copy the lines below into your input, between '* xyz charge multiplicity' and '*':\n")
print(coords)

view = py3Dmol.view(width=600, height=400)
view.addModel(f"{mol.GetNumAtoms()}\n\n{coords}\n", "xyz")
view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
view.zoomTo()
view.show()

## Notes

- The structure comes from a **force field** (MMFF). It is a reasonable start structure,
  not a final one: **optimise it with ORCA** (`Opt`) before you use it.
- For flexible molecules this is just *one* conformer, not necessarily the lowest. Use a
  conformer search (Exercise 6) to find the others.
- **Stereochemistry:** a name or SMILES without stereo information gives an arbitrary
  stereoisomer. Specify it, e.g. `trans-2-butene`, `(R)-2-butanol`, `C/C=C/C`,
  `C[C@@H](O)CC`.
- **Charge and multiplicity** are taken from the SMILES (e.g. `[NH4+]` → charge 1).
  Use them in the `* xyz` line of your input.
- Names are looked up in [PubChem](https://pubchem.ncbi.nlm.nih.gov). If a name is not
  found, try another name or draw the molecule in PubChem and copy its SMILES.